In [2]:
from ultralytics import YOLO
import torch
import os
import pandas as pd
import matplotlib.pyplot as plt


In [3]:
DATA_YAML_PATH = r"/home/student/SonarDataModel/sonar_dataset_10k_3k_3k.yaml"
MODEL_NAME = "yolo11s.pt"

IMG_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 100
WORKERS = 8

PROJECT_DIR = "runs/detect"
RUN_NAME = "yolo_full_100_epochs"

In [3]:
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------
# Load Model
# --------------------------------
model = YOLO(MODEL_NAME)

# --------------------------------
# Train (NO resume, clean run)
# --------------------------------
results = model.train(
    data=DATA_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=0,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=False,        # force new clean run
    save=True,
    pretrained=True,
    optimizer="auto",
    lr0=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    box=7.5,
    cls=0.5,
    dfl=1.5,
)

print("Training Complete")

# --------------------------------
# Get correct run directory
# --------------------------------
RUN_DIR = results.save_dir
print("Run Directory:", RUN_DIR)

results_csv_path = os.path.join(RUN_DIR, "results.csv")

# --------------------------------
# Verify CSV
# --------------------------------
if not os.path.exists(results_csv_path):
    raise FileNotFoundError("results.csv not found!")

df = pd.read_csv(results_csv_path)

print("Total epochs logged:", len(df))
print("Columns:")
print(df.columns.tolist())

# --------------------------------
# Ensure correct columns exist
# --------------------------------
required_columns = [
    "epoch",
    "time",
    "train/box_loss",
    "train/cls_loss",
    "train/dfl_loss",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
    "val/box_loss",
    "val/cls_loss",
    "val/dfl_loss",
    "lr/pg0",
    "lr/pg1",
    "lr/pg2",
]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    print("WARNING: Missing columns:", missing)

# --------------------------------
# Plotting (FULL 1–100)
# --------------------------------
PLOT_DIR = os.path.join(RUN_DIR, "custom_plots")
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(fig, name):
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)

# 1️⃣ Loss Curves
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], df["train/box_loss"], label="Train Box")
ax.plot(df["epoch"], df["val/box_loss"], label="Val Box")
ax.plot(df["epoch"], df["train/cls_loss"], label="Train CLS")
ax.plot(df["epoch"], df["val/cls_loss"], label="Val CLS")
ax.plot(df["epoch"], df["train/dfl_loss"], label="Train DFL")
ax.plot(df["epoch"], df["val/dfl_loss"], label="Val DFL")
ax.set_title("Train vs Val Loss (All Epochs)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True)
save_plot(fig, "loss_curves.png")

# 2️⃣ mAP
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
ax.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
ax.set_title("mAP over Epochs")
ax.set_xlabel("Epoch")
ax.set_ylabel("mAP")
ax.legend()
ax.grid(True)
save_plot(fig, "map_curves.png")

# 3️⃣ Precision & Recall
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], df["metrics/precision(B)"], label="Precision")
ax.plot(df["epoch"], df["metrics/recall(B)"], label="Recall")
ax.set_title("Precision & Recall over Epochs")
ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.legend()
ax.grid(True)
save_plot(fig, "precision_recall.png")

# 4️⃣ Learning Rate
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], df["lr/pg0"], label="lr/pg0")
ax.plot(df["epoch"], df["lr/pg1"], label="lr/pg1")
ax.plot(df["epoch"], df["lr/pg2"], label="lr/pg2")
ax.set_title("Learning Rate Schedule")
ax.set_xlabel("Epoch")
ax.set_ylabel("LR")
ax.legend()
ax.grid(True)
save_plot(fig, "learning_rate.png")

print("All plots generated for full 1–100 epochs.")


Device: CUDA
GPU: NVIDIA RTX A4000
Ultralytics 8.4.14 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX A4000, 15968MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/student/SonarDataModel/sonar_dataset_10k_3k_3k.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_full_100_epochs2, nbs=64, nms=False, ops

In [5]:
# --------------------------------
# Load Best Saved Model
# --------------------------------
BEST_MODEL_PATH = os.path.join('/home/student/SonarDataModel/best.pt')

if not os.path.exists(BEST_MODEL_PATH):
    raise FileNotFoundError("best.pt not found!")

print("Loading model from:", BEST_MODEL_PATH)
model = YOLO(BEST_MODEL_PATH)

# --------------------------------
# Validation on 3000 Validation Images
# --------------------------------
print("\nRunning Validation on 3000 validation images...")

val_results = model.val(
    data=DATA_YAML_PATH,   # or "sonar_eval.yaml"
    split="val",           # explicitly validate on val folder
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    save_json=True,
    save_hybrid=True,
    project=PROJECT_DIR,
    name="validation_3000",
    exist_ok=True
)

print("Validation Complete!")

# --------------------------------
# Testing on 3000 Test Images
# --------------------------------
print("\nRunning Testing on 3000 test images...")

test_results = model.val(
    data=DATA_YAML_PATH,   # or "sonar_eval.yaml"
    split="test",          # explicitly test on test folder
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    save_json=True,
    save_hybrid=True,
    project=PROJECT_DIR,
    name="test_3000",
    exist_ok=True
)

print("Testing Complete!")

Loading model from: /home/student/SonarDataModel/best.pt

Running Validation on 3000 validation images...
WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.4.14 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX A4000, 15968MiB)
YOLO11s summary (fused): 100 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2643.5±1678.5 MB/s, size: 125.8 KB)
val: Scanning /home/student/SonarDataModel/sonardataset_10k_3k_3k/val/labels.cache... 3000 images, 2030 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3000/3000 393.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 188/188 9.0it/s 21.0s0.1ss
                   all       3000       2173       0.98      0.914      0.953      0.814
                 MILCO        843       1420      0.977       0.89       0.94      0.787
                 NOMBO        392        753      0.984      0.938      0.